### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.

The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

In [4]:
import os
import openai
from dotenv import load_dotenv

## This function will load all the variable from .env file and will make them available
## os.environ directory (env_variablea)
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [5]:
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3.6-27b",
                    groq_api_key=groq_api_key)

In [6]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [7]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/Users/spy/Desktop/agentic-AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10139.73it/s]


#### VectorStore

In [9]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(documents=documents,
                                     embedding=embeddings)
vector_store

In [10]:
vector_store.similarity_search('cat')

[Document(id='c6bb882d-bb01-46ef-a8da-b5c43f31aff6', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='30b3140b-9972-44ac-8db2-dc7e7102e1b6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='3cb227b4-8da8-4a8f-aef7-e9a7b261763b', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='472736c0-ea43-4be9-bcc1-4c50983c73c0', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

### Async query

In [11]:
await vector_store.asimilarity_search('cat')

[Document(id='c6bb882d-bb01-46ef-a8da-b5c43f31aff6', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='30b3140b-9972-44ac-8db2-dc7e7102e1b6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='3cb227b4-8da8-4a8f-aef7-e9a7b261763b', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='472736c0-ea43-4be9-bcc1-4c50983c73c0', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [12]:
vector_store.similarity_search_with_score('cat')

[(Document(id='c6bb882d-bb01-46ef-a8da-b5c43f31aff6', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056218147278),
 (Document(id='30b3140b-9972-44ac-8db2-dc7e7102e1b6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='3cb227b4-8da8-4a8f-aef7-e9a7b261763b', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956907272338867),
 (Document(id='472736c0-ea43-4be9-bcc1-4c50983c73c0', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.665792465209961)]

### Retrievers

LangChain VectorStore objects do not subclass Runnable, and so cannot be immediately integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose the method we wish to use to retrieve documents, we can create a runnable easily. Below we will build...

In [13]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vector_store.similarity_search).bind(k=1)
retriever.batch(['cat', 'dog'])

[[Document(id='c6bb882d-bb01-46ef-a8da-b5c43f31aff6', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='30b3140b-9972-44ac-8db2-dc7e7102e1b6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

VectorStores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them.

For instance, we can replicate the above with the following:

In [14]:
### best way
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':1}
)

retriever.batch(['cat', 'dog'])

[[Document(id='c6bb882d-bb01-46ef-a8da-b5c43f31aff6', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='30b3140b-9972-44ac-8db2-dc7e7102e1b6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
    Answer the  question using  the context provided:
    
    {question}
    
    Context:
    {context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("human", message)
    ]
)  

rag_chain = {"context":retriever,
             'question':RunnablePassthrough()} | prompt | model 

response = rag_chain.invoke("tell me about dogs")
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "tell me about dogs"
   - **Context:** `[Document(id='30b3140b-9972-44ac-8db2-dc7e7102e1b6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]`

2.  **Identify Key Constraints:**
   - The answer must be based *only* on the provided context.
   - The context explicitly states: "Dogs are great companions, known for their loyalty and friendliness."

3.  **Formulate Response:**
   - Directly extract the relevant information from the context.
   - Keep it concise and aligned with the prompt.
   - Draft: Based on the provided context, dogs are great companions that are known for their loyalty and friendliness.

4.  **Check Against Constraints:**
   - Does it answer the question? Yes.
   - Is it solely based on the context? Yes.
   - Does it add external information? No.
   - Is it clear and concise? Yes.

5.  **Final Output Generation